In [1]:
import os
import sys
import pickle
sys.path.append(os.path.abspath('..'))

import numpy as np
from src.qulacs_vqe import run_hva_vqe, sample_hva_portfolios, best_sampled_portfolios, run_hea_vqe, sample_hea_portfolios
from src.plotting import plot_vqe_convergence, plot_portfolio_objective_histogram
from src.qubo import qubo_value

In [2]:
with open('results/raw/qubo_ising_data.pkl', 'rb') as f:
    data_n2 = pickle.load(f)

K, g = data_n2['K'], data_n2['g']
q, Q, const = data_n2['q'], data_n2['Q'], data_n2['const']
print("Data QUBO dan Ising berhasil di-load!")

Data QUBO dan Ising berhasil di-load!


In [ ]:
print("Memulai optimasi VQE (Ansatz: HVA)...")
vqe_res_hva = run_hva_vqe(K, g, gamma_tf=0.2, depth=2)
print("HVA VQE Energy:", vqe_res_hva.fun)

print("Plotting HVA Convergence:")
plot_vqe_convergence(vqe_res_hva.energy_history)

In [ ]:
print("Memulai optimasi VQE (Ansatz: HEA)...")
depth_hea = 2
vqe_res_hea = run_hea_vqe(K, g, gamma_tf=0.2, depth=depth_hea)
print("HEA VQE Energy:", vqe_res_hea.fun)

print("Plotting HEA Convergence:")
plot_vqe_convergence(vqe_res_hea.energy_history)

In [ ]:
n_shots = 2000
samples_hva = sample_hva_portfolios(K, g, vqe_res_hva.x, n_shots=n_shots)
top_quantum_hva = best_sampled_portfolios(samples_hva, q, Q, const, top_k=5)

print("\n--- Top 5 Portofolio Quantum (HVA) ---")
for idx, (bitstring, obj_val) in enumerate(top_quantum_hva):
    selected_actions = sum(bitstring)
    print(f"Rank {idx+1} | Total Aksi: {selected_actions} | Objective: {obj_val:.4f}")

# Calculate objective values for all samples to plot histogram
sampled_objs_hva = [qubo_value(x, q, Q, const) for x in samples_hva]
plot_portfolio_objective_histogram(sampled_objs_hva)

In [ ]:
with open('results/raw/vqe_results.pkl', 'wb') as f:
    pickle.dump({
        'best_energy_hva': vqe_res_hva.fun, 
        'top_portfolios_hva': top_quantum_hva,
        'best_energy_hea': vqe_res_hea.fun
    }, f)
print("\nHasil eksekusi VQE berhasil disimpan ke pickle!")